<a href="https://colab.research.google.com/github/AustinBramlett/movie-recommender-svd/blob/main/notebooks/02_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

def split_ratings(ratings, test_frac=0.10, seed=0):
    rng = np.random.default_rng(seed)
    is_test = np.zeros(len(ratings), dtype=bool)

    user_left = ratings.groupby("user").size().to_dict()
    item_left = ratings.groupby("item").size().to_dict()

    for pos in rng.permutation(len(ratings)):
        row = ratings.iloc[pos]
        user, item = row["user"], row["item"]

        if (rng.random() < test_frac
                and user_left[user] > 1
                and item_left[item] > 1):
            is_test[pos] = True
            user_left[user] -= 1
            item_left[item] -= 1

    return ratings.iloc[~is_test].copy(), ratings.iloc[is_test].copy()


train, test = split_ratings(ratings)

print("Training ratings:", len(train))
print("Testing ratings:", len(test))
print("Test fraction:", round(len(test) / len(ratings), 3))


This code splits the ratings into two seperate data values. About 10% is used for testing. Each user and movie keeps at least one rating in the training data.

In [ ]:

num_users = ratings["user"].max() + 1
num_items = ratings["item"].max() + 1

R = np.zeros((num_users, num_items))

for _, row in train.iterrows():
    R[int(row["user"]), int(row["item"])] = row["rating"]

print("Matrix shape:", R.shape)
print("Nonzero ratings:", np.count_nonzero(R))


This creates a matrix using the training data each row represents a user and each column represents a movie. All of the Missing ratings are being set to zero.

In [ ]:

def rmse(actual, predicted):
    return np.sqrt(np.mean((actual - predicted) ** 2))


print("RMSE test:", rmse(np.array([3, 4]), np.array([3, 5])))


This function measures how far the predicted ratings are from the actual ratings. It uses the RSME for predictions

In [ ]:

global_mean = train["rating"].mean()

user_means = train.groupby("user")["rating"].mean()
item_means = train.groupby("item")["rating"].mean()

global_pred = np.full(len(test), global_mean)
user_pred = test["user"].map(user_means).fillna(global_mean).to_numpy()
item_pred = test["item"].map(item_means).fillna(global_mean).to_numpy()

print("Global mean RMSE:", rmse(test["rating"].to_numpy(), global_pred))
print("User mean RMSE:", rmse(test["rating"].to_numpy(), user_pred))
print("Movie mean RMSE:", rmse(test["rating"].to_numpy(), item_pred))


This program creates three different models... One predicts the overall average rating and then one uses the users average and one uses each moviess average. RMSE compares how accurate the predictions are.

In [ ]:

user_bias = train.groupby("user")["rating"].mean() - global_mean
item_bias = train.groupby("item")["rating"].mean() - global_mean

pred = (
    global_mean
    + test["user"].map(user_bias).fillna(0)
    + test["item"].map(item_bias).fillna(0)
)

print("User + movie effects RMSE:", rmse(test["rating"].to_numpy(), pred.to_numpy()))


This one combines the overall average rating with each user and movie rating differences. RMSE measures how accurate the predictions are.

In [ ]:

import numpy as np

R = np.array([
    [5, 3, np.nan],
    [4, np.nan, 1],
    [np.nan, 2, 2]
])


mu = np.nanmean(R)


user1 = np.nanmean(R[0]) - mu
user2 = np.nanmean(R[1]) - mu
user3 = np.nanmean(R[2]) - mu

print("Overall average:", mu)
print("User biases:", user1, user2, user3)


b_user = np.array([user1, user2, user3])
movie_bias = np.zeros(3)

for i in range(3):
    total = 0
    count = 0

    for j in range(3):
        if not np.isnan(R[j][i]):
            total += R[j][i] - mu - b_user[j]
            count += 1

    if count > 0:
        movie_bias[i] = total / count

print("Movie biases", movie_bias)


In [ ]:

R_all = to_matrix(ratings)

user_means = np.nanmean(R_all, axis=1)

pred = user_means[u]

score = rmse(pred, truth)

print("RMSE:", score)


This code will calculate each user's average rating using all the data it will then checks how accurate the predictions are. The score is misleading because it uses ratings that shouldn't be seen.

In [ ]:

train, test = split_ratings(ratings, seed=1)

mu = train["rating"].mean()

user_means = train.groupby("user")["rating"].mean()
item_means = train.groupby("item")["rating"].mean()

actual = test["rating"].to_numpy()

pred1 = np.full(len(test), 3.0)
pred2 = np.full(len(test), mu)
pred3 = test["user"].map(user_means).fillna(mu).to_numpy()
pred4 = test["item"].map(item_means).fillna(mu).to_numpy()

user_bias = user_means - mu
residual = train["rating"] - mu - train["user"].map(user_bias)
item_bias = residual.groupby(train["item"]).mean()

pred5 = mu + test["user"].map(user_bias).fillna(0) + test["item"].map(item_bias).fillna(0)
pred5 = pred5.clip(1, 5).to_numpy()

print("Always 3:", rmse(actual, pred1))
print("Overall average:", rmse(actual, pred2))
print("User average:", rmse(actual, pred3))
print("Movie average:", rmse(actual, pred4))
print("User and movie:", rmse(actual, pred5))


This code repeats the test with the different seed and calculates all five of the RMSE scores. The scores may change because the ratings are split differentlythan the others

The RMSE values can change with a different seed The model rankings might change as well so the models should be compared using the same test data.